# Scube Mouse Hypothalamus (2/2): Alignment of consecutive ST slices

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [2]:
import SPACEL
from SPACEL import Scube

In [3]:
seed = 2
from SPACEL.setting import set_environ_seed
set_environ_seed(seed)

Setting environment seed: 2


## Spatial transcriptomic data import

The input data type of Scube is a list of anndata objects for each slice. The anndata objects of each slice must contain a column of spatial domain for alignment in ``.obs``.

In [ ]:
adata = sc.read(f'./{seed}/HBC_Splane.h5ad')

adata[0].obs

,cell_id,transcript_counts,control_probe_counts,control_codeword_counts,total_counts,cell_area,nucleus_area,region,dataset,celltype_major,...,Myoepithelial,NK cells,NKT cells,PVL Differentiated,PVL Immature,Plasmablasts,T cells CD4+,T cells CD8+,slice_id,spatial_domain
0,1,28,1,0,29,58.387031,26.642188,cell_circles,xe_rep1,Normal Epithelial,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,27


In [5]:
adata_list=[]

for i in range(2):
    adata_list.append(adata[adata.obs['slice_id'] == i])
    adata_list[i].obs.spatial_domain = adata_list[i].obs.spatial_domain.astype('category')

In [6]:
adata_list[1]

AnnData object with n_obs × n_vars = 118752 × 313
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'dataset', 'celltype_major', 'celltype_minor', 'original_clusters', 'B cells Memory', 'B cells Naive', 'CAFs MSC iCAF-like', 'CAFs myCAF-like', 'Cancer Basal SC', 'Cancer Cycling', 'Cancer Her2 SC', 'Cancer LumA SC', 'Cancer LumB SC', 'Cycling PVL', 'Cycling T-cells', 'Cycling_Myeloid', 'DCs', 'Endothelial ACKR1', 'Endothelial CXCL12', 'Endothelial Lymphatic LYVE1', 'Endothelial RGS5', 'Luminal Progenitors', 'Macrophage', 'Mature Luminal', 'Monocyte', 'Myoepithelial', 'NK cells', 'NKT cells', 'PVL Differentiated', 'PVL Immature', 'Plasmablasts', 'T cells CD4+', 'T cells CD8+', 'slice_id', 'spatial_domain'
    obsm: 'spatial'

Plot all raw slices stacked in a single figure. Cells colored by spatial domain.First, we define the color of each spatial domain.

## Pairwise alignment

Here, we perform pairwise alignment of slices. The ``cluster_key`` is a column of ``.obs`` which contain spatial domain information used for alignment. The ``n_neighbors`` affect how much neighbors are considered when calculate similarity of two slices. The ``knn_exclude_cutoff`` affect max distance of neighbors. By default, ``n_neighbors = 15`` and ``knn_exclude_cutoff = n_neighbors`` means a spot/cell in source slices consider 15 neighbors in target slices, and neighbor will be exclude when the distance of neighbors larger than the median of neareast 30 neighbors distance in all spot/cell in target slice. ``p`` affect the exponent of overlap penalty, and a larger p means stronger partial alignment capability.

In [ ]:
Scube.align(adata_list,
      cluster_key='spatial_domain', 
      n_neighbors = 15, 
      n_threads=10,
      p=2,
      seed=seed,
     )

Start alignment...
Alignment slice 1 to 0
Runtime: 2342.4375743865967 s


After alignment, the new spots/cells location coordinates saved in ``.obsm['spatial_aligned']`` of anndata object of each slices. When ``write_loc_path`` provied by users, the location coordinates of all spots/cells of all slices will be saved in ``write_loc_path``.

## LAA & ARI

In [8]:
from scipy.spatial import cKDTree
from sklearn.metrics import adjusted_rand_score


def find_nearest_points(f1, f2):
    """
    Find the nearest point in f2 for each point in f1 using batch processing.
    
    Args:
        f1 (numpy.ndarray): Array of shape (n_points_1, 2) representing points in f1.
        f2 (numpy.ndarray): Array of shape (n_points_2, 2) representing points in f2.
        batch_size (int): Number of points to process in each batch.
        
    """
    # Build a k-d tree for f2
    tree = cKDTree(f2)
    
    # Query the k-d tree for nearest neighbors of points in f1
    distances, nearest_indices = tree.query(f1)
        
    cnt = 0
    true_clusters_mapped_region = []
    mapped_clusters_mapped_region = []

    for i in range(len(f1)):
        f1_cluster = adata_list[0].obs.iloc[i]['original_clusters']
        f2_cluster = adata_list[1].obs.iloc[nearest_indices[i]]['original_clusters']
        true_clusters_mapped_region.append(f1_cluster)
        mapped_clusters_mapped_region.append(f2_cluster)
        if f1_cluster == f2_cluster:
            cnt += 1 

    laa = cnt/len(f2)
    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
              
    return laa,ari


In [ ]:
ari_values = []
laa_values = []
slices = [0, 1]

for i in range(1, len(slices)):
    target_ind = i-1
    source_ind = i
    f1 = adata_list[target_ind].obsm['spatial_aligned'].to_numpy()
    f2 = adata_list[source_ind].obsm['spatial_aligned'].to_numpy()

    laa,ari = find_nearest_points(f1, f2)
    ari_values.append(ari)
    laa_values.append(laa)
    print(laa, ari)
    
# Create DataFrame with desired format
header_row = [f"{slices[i-1]}_{slices[i]}" for i in range(1, len(slices))]
ari_df = pd.DataFrame([ari_values], columns=header_row)
laa_df = pd.DataFrame([laa_values], columns=header_row)

0.08107652923740231 0.0006043057923311798


In [ ]:
# Save as a CSV file
ari_df.to_csv(f'./{seed}/HBC_rep1_rep2_ARI.csv')
laa_df.to_csv(f'./{seed}/HBC_rep1_rep2_LAA.csv')

## Plot alignment results

Save all spots/cells aligned coordiantes and colors in a DataFrame.

In [11]:
c_dict = {'B cells Memory': '#1f77b4',
 'B cells Naive': '#aec7e8',
 'CAFs MSC iCAF-like': '#ff7f0e',
 'CAFs myCAF-like': '#ffbb78',
 'Cancer Basal SC': '#2ca02c',
 'Cancer Cycling': '#98df8a',
 'Cancer Her2 SC': '#d62728',
 'Cancer LumA SC': '#ff9896',
 'Cancer LumB SC': '#9467bd',
 'Cycling PVL': '#c5b0d5',
 'Cycling T-cells': '#8c564b',
 'Cycling_Myeloid': '#c49c94',
 'DCs': '#e377c2',
 'Endothelial ACKR1': '#f7b6d2',
 'Endothelial CXCL12': '#7f7f7f',
 'Endothelial Lymphatic LYVE1': '#c7c7c7',
 'Endothelial RGS5': '#bcbd22',
 'Luminal Progenitors': '#dbdb8d',
 'Macrophage': '#17becf',
 'Mature Luminal': '#9edae5',
 'Monocyte': '#393b79',
 'Myoepithelial': '#5254a3',
 'NK cells': '#6b6ecf',
 'NKT cells': '#9c9ede',
 'PVL Differentiated': '#637939',
 'PVL Immature': '#8ca252',
 'Plasmablasts': '#b5cf6b',
 'T cells CD4+': '#cedb9c',
 'T cells CD8+': '#8c6d31'}

In [ ]:
coo = pd.DataFrame()

for i in range(len(adata_list)):
    loc = adata_list[i].obsm['spatial_aligned'].copy()
    loc['Z'] = i
    loc['slice'] = adata_list[i].obs['slice_id']
    loc['original_clusters'] = adata_list[i].obs['original_clusters']
    loc['celltype_colors'] = adata_list[i].obs['original_clusters'].replace(c_dict)
    coo = pd.concat([coo,loc],axis=0)
    
coo.to_csv(f'./{seed}/SPACEL_HBC_labels_aligned_coordinates_colors.csv')

In [13]:
coo

,X,Y,Z,slice,original_clusters,celltype_colors
0,-1967.726685,-2263.500977,0,0,Myoepithelial,#5254a3
1,-1988.644531,-2261.660645,0,0,Cancer LumA SC,#ff9896
2,-1966.219727,-2257.949219,0,0,Endothelial RGS5,#bcbd22
3,-1990.758179,-2255.439697,0,0,Plasmablasts,#b5cf6b
4,-1973.629028,-2257.449951,0,0,Cancer LumA SC,#ff9896
...,...,...,...,...,...,...
118747,3981.129150,2943.008789,1,1,Macrophage,#17becf
118748,3963.196045,2946.398193,1,1,Endothelial RGS5,#bcbd22
118749,3988.519775,2956.489746,1,1,Endothelial RGS5,#bcbd22
118750,3971.567627,2960.873535,1,1,CAFs MSC iCAF-like,#ff7f0e


In [ ]:
def plot_3d(
    loc,
    val=None,
    color=None,
    figsize=(12,12),
    return_fig=True,
    elev=None,
    azim=None,
    roll=None,
    xlim=None,
    ylim=None,
    zlim=None,
    frameon=True,
    save_path=None,
    save_dpi=150,
    show=True,
    *args,
    **kwargs
):
    """Plot all slices stacked in 3D

    Plot all slices stacked with a given number of subplots rows and columns. Spots/cells colored by spatial domain.

    Args:
        loc: An array of the coordinates of each spots/cells in all slices, which the first three columns are X-axis, Y-axis, Z-axis coordinates.
        val: The colors of each spots/cells given in loc use to plot.
        color: The colors of each spots/cells given in loc use to plot.
        figsize: Size of the figure.
        return_fig: Whether to return the figure.
        elev: The elevation angle in the vertical plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        azim: The azimuth angle in the horizontal plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        xlim: A tuple given the left and right xlims in X-axis.
        ylim: A tuple given the left and right xlims in Y-axis.
        zlim: A tuple given the left and right xlims in Z-axis.
        frameon: Whether to hide the coordinate axes.
        save_path: A string representing the path directory where the figure saved.
        save_dpi: The resolution in dots per inch.
        show: Whether to show the figure.

    Returns:
        A ``matplotlib.figure.Figure`` object
    """
    if 'marker' not in kwargs.keys():
        kwargs['marker'] = 'o'
    if 's' not in kwargs.keys():
        kwargs['s'] = 5
    if 'cmap' not in kwargs.keys():
        kwargs['cmap'] = 'Spectral_r'
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection='3d')
    if (elev is not None) and (azim is not None) and (roll is not None):
        ax.view_init(elev, azim, roll)
    if color is None:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=val,*args,**kwargs)
    else:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=color,*args,**kwargs)
    
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_box_aspect([2, 2, 1])
    if not frameon:
        plt.axis('off')
        plt.grid(b=None)
    if save_path is not None:
        print(save_path)
        plt.savefig(save_path,dpi=save_dpi,bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    if return_fig:
        return fig

In [ ]:
seed = 2024
df = pd.read_csv(f'./{seed}/SPACEL_HBC_labels_aligned_coordinates_colors.csv')
coords = df[['X', 'Y', 'Z']]

In [ ]:
fig = plot_3d(coords.values, val=None, color=df['celltype_colors'], elev=30, azim=20, roll=0, frameon=False, show=True, s=0.5, save_dpi=300, return_fig=True) # Scube is from SPACEL

fig.savefig(f'./{seed}/SPACEL_gtAndScube_HBC_labels_3dviz_angle30_20_s=0.5.png', bbox_inches="tight")